# **BAMS 508 Final Project：**
## **Option 2. Pediatrician Scheduling**

Shizhuang (Chris) Guo 72435357 <br>
Shuyi (Rebecca)	Peng 84163724 <br>
Novar Qin 92408798 <br>
Irene Yu 30709356 <br>
Michael Ruizhe Zhang 82889767 <br>


## Executive Summary

For the Pediatrician Scheduling Problem, we outlined the hard and soft constraints in an algebraic model based on the case information (Part A) and implemented it in Gurobi to find optimal solutions (Part C). The model uses data from a four-week scheduling cycle, which can be updated for future cycles to ensure ongoing relevance.

In Part B, we highlighted some potential drawbacks of relying solely on generative AI tools like ChatGPT. While these tools can be helpful, they might cause inefficiencies in designing the model and make it tricky to properly formulate more complex soft constraints.

In Part D, we took a step back to consider the real-world implications of the problem, focusing on how the model could be made more flexible to adjust hard and soft constraints as needed. This flexibility in model design is key to handling changes in scheduling demands and management preferences.

## Introduction

The case study, Pediatrician Scheduling at British Columbia Women’s Hospital, provides the background for the problem at hand. Our aim is to develop a solution for Meg to address the challenges in scheduling pediatricians. Specifically, our goal is to create a scheduling solution that meets all constraints outlined in the case study, using linear programming and Gurobi to automate the scheduling process. Applying our knowledge from BAMS 508, we aim to streamline operations and improve efficiency in the hospital’s pediatrician scheduling system.

## Part A. Model Formulation

### 1. Decision Variables

- $x_{p,s} \in \{0, 1\}$: A binary variable indicating if pediatrician $p$ works on shift $s$.
- $pow_{p,w} \in \{0, 1\}$: A binary variable indicating if pediatrician $p$ is assigned a POW during week $w$.
- $weekend\_work_{p, w} \in \{0, 1\}$: A binary variable indicating if pediatrician $ p $ works during weekend $ w $.


- Violation variables for soft constraints:
  - $SC1_{p,w}$: violation of consecutive weekend shifts, indicating if pediatrician $p$ works two consecutive weekends.
  - $SC2_{p,s}$: violation of night shift spacing, indicating if pediatrician $p$ works two night shifts on consecutive days.
  - $SC3_{p,w}$: violation of evening shift limits, indicating if pediatrician $p$ exceeds the maximum allowed evening shifts in week $w$.
  - $SC4_{p}$: violation of day/night balance, indicating if pediatrician $p$'s assignments are unbalanced between day and night shifts.


### 2. Constraints

a. **Shift Coverage**:
   - Each shift on each day must be covered by exactly one pediatrician:
   $$
   \sum_{p=1}^{15} x_{p,s} = 1 \quad \forall \quad s \in \text{Shifts} = \{1,2,...,56\}
   $$

b. **Shift Requirements**:
   - Each pediatrician must work exactly the required number of shifts:
   $$
   \sum_{s=1}^{56} x_{p,s} = \text{required\_shifts}_p \quad \forall p \in \text{Pediatricians}
   $$

c. **Back-to-back Shift Constraints**:
   - No pediatrician can work consecutive shifts:
   $$
   x_{p,s} + x_{p,s+1} \leq 1 \quad \forall p \in \text{Pediatricians}, \quad s \in \text{Shifts} = \{1,2,...,55\}
   $$

d. **Consecutive Night Shift Limit**:
   - No pediatrician can work consecutive night shifts:
   $$
   x_{p,s} + x_{p,s+2} \leq 1 \quad \forall p \in \text{Pediatricians}, \quad s \in \text{Shifts} = \{2,4,6,...,54\}
   $$

e. **Weekend Shift Limit (Optional)**: When constraints c and d are satisfied, constraint e will always be met.
   - A pediatrician can work at most two shifts during the weekend (Friday night to Sunday night):
   $$
   \sum_{s \in \text{weekend\_shifts}} x_{p,s} \leq 2 \quad \forall p \in \text{Pediatricians},
   \\
   \quad s \in \text{Weekend Shifts} = \{(w-1)\cdot14+9,\quad (w-1)\cdot14+10,\quad (w-1)\cdot14+11,\quad (w-1)\cdot14+12\, \quad (w-1)\cdot14+13\}   \quad w \in {1, 2, 3, 4}
   $$

f. **Availability Constraints**:
   - For each pediatrician $p$ on shift $s$, they can only work if they are available for that shift. This is represented as:
   $$
   x_{p,s} \leq availability_{s, p}
   $$

g. **POW Assignment Constraints**:
   - A pediatrician must be assigned exactly one POW per week:
   $$
   \sum_{p=1}^{15} pow_{p,w} = 1 \quad \forall w \in \text{Weeks} = \{1,2,3,4\}
   $$

h. **POW Mandatory Shifts Constraints**:
   - The POW must cover (at least) the following shifts: Tuesday day, Thursday day, and Saturday night:
   $$
   x_{p,3 + (w-1)\cdot14} + x_{p,7 + (w-1)\cdot14} + x_{p,12 + (w-1)\cdot14}\geq 3\cdot pow_{p,w} \quad \forall p, ; w \in {1, 2, 3, 4}
   $$

i. **POW Monthly Limits**:
   - No one serves as the POW more than once in a four-week cycle:
   $$
   \sum_{w=1}^{4} pow_{p,w} <= 1 \quad \forall p \in \text{Pediatricians}
   $$

j. **POW Remaining Limits**:
   - Each pediatrician has a designated number of times they should be the POW over the course of the year:
   $$
   \sum_{w=1}^{4} pow_{p,w} <= remaining\_pow_{p} \quad \forall p \in \text{Pediatricians}
   $$

### 3. Soft Constraints:
a. **No consecutive weekends**:
   - Violation Definition:
   A violation occurs when a pediatrician works during two consecutive weekends. This is modeled as:

   $$
   weekend\_work_{p, w} + weekend\_work_{p, w+1} \leq 1 + consec\_weekend\_violations_{p, w}, \quad \forall p, \forall w \in {0, \dots, \text{NUM\_WEEKS}-2}
   $$

   - Weekend Work Determination:
   $$
   weekend\_work_{p, w} \leq \sum_{s \in \text{weekend\_shifts}_w} x_{p, s}, \quad \forall p, \forall w \text{optional}
   $$
   $$
   weekend\_work_{p, w} \geq x_{p, s}, \quad \forall p, \forall w, \forall s \in \text{weekend\_shifts}_w
   $$
   - If $weekend\_work_{p, w} = 1$, at least one shift $s$ in $\text{weekend\_shifts}_w$ must be assigned to pediatrician $p$.
   
   - If no weekend shifts are assigned ($\sum_{s \in \text{weekend\_shifts}_w} = 0$), then $weekend\_work_{p, w}$ must equal 0. This constraint is **Optional**, because the minimization process will automaticly make it happen.


b. **Night shift spacing**:
   A violation occurs if a pediatrician works night shifts spaced two days apart:
   $$
   x_{p,s} + x_{p,s+4} \leq 1 + night\_spacing\_violations_{p,s} \quad \forall p \in \text{Pediatricians}, \quad s \in \text{Shifts} = \{2,4,6...,52\}
   $$
   - Here,  $s+4$  refers to the next sequential shift after 2 days. Ensure that indices wrap correctly for the last shift of the scheduling period. 

c. **Evening shift limit**:
   A violation occurs if a pediatrician exceeds the maximum allowed evening shifts in a week:
   $$
   \sum_{s \in \text{week}} x_{p,s} \leq 2 + evening\_shift\_violations_{p,w} \quad \forall p \in \text{Pediatricians}, \quad w \in \text{Weeks}
   $$
   - week is the set of all shift indices in the given week.  

d. **Day/Night balance**:
   A violation occurs if the total number of shifts is imbalanced within a given scheduling period:
   $$
   \sum_{s \in \text{first\_half}} x_{p,s} - \sum_{s \in \text{second\_half}} x_{p,s} \leq day\_night\_balance\_violations_{p}
   $$
   and
   $$
   \sum_{s \in \text{second\_half}} x_{p,s} - \sum_{s \in \text{first\_half}} x_{p,s} \leq day\_night\_balance\_violations_{p}
   $$
   - first_half and second_half are predefined sets of shift indices, representing the first and second halves of the scheduling period (or some other division, depending on your problem structure).

### 4, Objective Function

The objective is to minimize the sum of violations for soft constraints:
$$
\text{minimize} \quad W1\times \sum_{p=1}^{15} \sum_{w=1}^{4} consec\_weekend\_violations_{p,w} + 
\\
W2\times \sum_{p=1}^{15} \sum_{s \in \text{Shifts}} night\_spacing\_violations_{p,s} + 
\\
W3\times \sum_{p=1}^{15} \sum_{w=1}^{4} evening\_shift\_violations_{p,w} + 
\\
W4\times \sum_{p=1}^{15} day\_night\_balance\_violations_{p}
$$

$$
{W1, W2, W3, W4}={0.3, 0.3, 0.3, 0.10}
$$

Explaination of $W_i$:

- Based on the management preference, we can adjust the weight of each penalties.

- Some pediatrician has odd number shift requirement (i.e. 3 days), which inevitably violate the day/night balance. Therefore, we only set 0.1 weight for day/night balance penalty.


## Part B. ChatGPT’s response


### 1. Comparative Analysis of Decision Variables

| **Feature**                 | **ChatGPT’s Decision Variables**                               | **Our Decision Variables**                             |
|-----------------------------|-------------------------------------------------------------|-------------------------------------------------------|
| **Primary Variables**        | - $x_{i,j,k}$: Binary variable indicating if pediatrician $i$ is assigned to shift $j$ in week $k$.  <br> - $y_{i,k}$: Binary variable indicating if pediatrician $i$ is the POW in week $k$. | - $x_{p,s}$: Binary variable indicating if pediatrician $p$ works on shift $s$. <br> - $pow_{p,w}$: Binary variable indicating if pediatrician $p$ is assigned a POW during week $w$. <br> - $weekend\_work_{p,w}$: Binary variable indicating if pediatrician $p$ works during weekend $w$. |
| **Soft Constraint Variables** | None explicitly modeled; violations embedded in objective function via $c_{1,i,j,k}$, $c_{2,i,j,k}$, etc. | - $SC1_{p,w}$: Violation of consecutive weekend shifts. <br> - $SC2_{p,s}$: Violation of night shift spacing. <br> - $SC3_{p,w}$: Violation of evening shift limits. <br> - $SC4_{p}$: Violation of day/night balance. |
| **Aggregation of Violations** | Violations (e.g., consecutive weekends) embedded into decision variables like $c_{1,i,j,k}$, making them harder to track and interpret. | Explicit soft constraint violation variables $SC1$, $SC2$, $SC3$, $SC4$ that simplify tracking and allow clear optimization. |
| **Variable Indexing**        | - Three indices for $x_{i,j,k}$: Requires looping through **pediatrician ($i$)**, **shift ($j$)**, and **week ($k$)**.  <br> - Redundant indexing between variables like $x_{i,j,k}$ and $y_{i,k}$. | - Simplified indexing: $x_{p,s}$ reduces to two indices (pediatrician $p$, shift $s$). <br> - $SC1_{p,w}, SC2_{p,s}, SC3_{p,w}, SC4_{p}$ aggregate violations over relevant indices. |
| **Redundancy**               | - Redundancy between $x_{i,j,k}$ and $y_{i,k}$ when handling POW assignments. <br> - Violations like consecutive weekend shifts are indirectly modeled, leading to repeated logic. | - No redundant variables; $weekend\_work_{p,w}$ handles weekend assignments cleanly. <br> - Violation variables directly encapsulate soft constraint violations, reducing complexity. |

---

#### Summary of Differences

1. **Simplified Indexing in Our Model:**
   - ChatGPT uses $x_{i,j,k}$, requiring **three indices** (pediatrician, shift, and week) for all constraints and objective function components. This adds complexity and makes the model harder to interpret and optimize.
   - Our model reduces $x$ to $x_{p,s}$, limiting indexing to **pediatrician ($p$)** and **shift ($s$)**, which simplifies constraints and calculations.

2. **Explicit Soft Constraint Violations in Our Model:**
   - Our model introduces explicit violation variables ($SC1, SC2, SC3, SC4$), which directly represent each soft constraint. This enhances clarity and separates decision variables from violations.
   - ChatGPT embeds violations within decision variables like $c_{1,i,j,k}$, making it harder to track which constraints are being violated and by how much.

3. **Redundancy in ChatGPT’s Model:**
   - ChatGPT’s $x_{i,j,k}$ and $y_{i,k}$ have overlapping functionality, especially for determining POW assignments. This redundancy can lead to unnecessary computations.
   - Our model eliminates redundancy by introducing $pow_{p,w}$ and $weekend\_work_{p,w}$, which handle specific tasks efficiently without overlap.

4. **Aggregation of Violations:**
   - ChatGPT’s violations are embedded within indexed variables (e.g., $c_{1,i,j,k}$), requiring additional summations to calculate total violations (e.g., consecutive weekends or shift spacing).
   - Our model aggregates violations at higher levels (e.g., $SC1_{p,w}$), avoiding repetitive calculations and improving model interpretability.

5. **Why Avoid Three-dimensional Variables in Gurobi:**
   - **Simplifying the Model:** 
     - Three-dimensional variables can be simplified into two-dimensional variables to reduce model complexity. For example, $x_{i,j,k}$ (representing a pediatrician $i$, shift $j$, and week $k$) is replaced with $x_{p,s}$ (representing a pediatrician $p$ and shift $s$). This avoids the need for explicitly tracking weeks in the decision variables.
     - Aggregated variables like $weekend\_work_{p,w}$ further reduce dimensionality by summarizing shift-level information at the week level.
   - **Advantages of Our Model:**
     - We replaced $x_{i,j,k}$ with $x_{p,s}$ to directly represent a pediatrician $p$ working a shift $s$, eliminating the week dimension ($k$).
     - Aggregated variables (e.g., $weekend\_work_{p,w}$ and $SC1_{p,w}$) handle complex states like weekend assignments and violations, further improving efficiency.
     - **Improved Results:**
       - **Reduced Variable Dimensions:** The three-dimensional variables $i,j,k$ are reduced to two-dimensional variables $p,s$, while auxiliary variables independently handle weekly or shift-specific constraints.
       - **Simplified Constraint Expressions:** The logic of constraints no longer requires nested indexing and can be expressed with two-layer loops.
       - **Optimized Efficiency:** By reducing dimensions, Gurobi becomes more efficient in solving the problem, making the model easier to debug and maintain.


---

#### Conclusion

- **ChatGPT’s Decision Variables:**
  - Correct and functional but overly complex due to redundant variables and lack of explicit aggregation.
  - Requires more indexing and summation, which can lead to computational inefficiencies, particularly in Gurobi.

- **Our Decision Variables:**
  - Simpler and more structured with explicit violation variables ($SC1, SC2, SC3, SC4$).
  - Aggregated variables like $weekend\_work_{p,w}$ reduce redundancy, improve clarity, and align better with Gurobi’s optimization capabilities by avoiding unnecessary three-dimensional variables.



### 2. Soft Constraint Comparison for Work Consecutive Weekends

| **Feature**                  | **ChatGPT’s Model**                                                                 | **Our Model**                                                                                           |
|------------------------------|-----------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------|
| **Violation Variable**        | $c_{2,i,j,k}$: Binary variable indicating violations of night shift spacing for pediatrician $i$ between shifts $j$ and $j+1$. | $SC2_{p,s}$: Binary variable indicating violations of night shift spacing for pediatrician $p$ on shift $s$. |
| **Shift Spacing Logic**       | Checks consecutive night shifts using $c_{2,i,j,k}$, tied directly to $x_{i,j,k}$. | Prevents consecutive night shifts by ensuring sufficient spacing between shifts at the aggregated level. |
| **Dimensionality**            | Relies on three indices ($i, j, k$).                                                | Reduced to two indices ($p, s$), simplifying the representation and constraint expressions.            |
| **Violation Representation** | Embedded in the objective function through $c_{2,i,j,k}$, requiring nested evaluations. | Explicitly defined as $SC2_{p,s}$, making violations clear and separate from decision variables.       |
| **Boundary Handling**         | Relies on optimization to handle boundary shifts; no explicit constraints for last shifts. | Explicit constraints ensure boundary conditions (e.g., $s+1$ and $s+2$) are managed logically.          |
| **Model Robustness**          | May lead to ambiguity or inefficiencies in handling end-of-week or irregular schedules. | Robust and consistent with clear violation representation and explicit constraints for shift spacing. |

---

#### Summary of Key Differences

1. **Dimensionality**:
   - **ChatGPT**: Relies on three-dimensional indexing ($i, j, k$), making constraints harder to interpret and computationally more expensive.
   - **Our Model**: Reduces to two dimensions ($p, s$), simplifying the model and improving efficiency.

2. **Violation Representation**:
   - **ChatGPT**: Embeds violations directly in $c_{2,i,j,k}$, which is tied to $x_{i,j,k}$. This leads to less clear separation of decision variables and constraints.
   - **Our Model**: Uses $SC2_{p,s}$ to explicitly track violations, making the logic clear and separate from the decision variables.

3. **Boundary Handling**:
   - **ChatGPT**: Lacks explicit constraints for managing boundary shifts, relying on implicit optimization to handle edge cases.
   - **Our Model**: Provides explicit constraints for boundaries like $s+1$ and $s+2$, ensuring logical consistency.

4. **Robustness**:
   - **ChatGPT**: May lead to inefficiencies or ambiguities in scenarios like end-of-week or irregular schedules.
   - **Our Model**: Robustly handles all cases with clear violation definitions and explicit constraints.

---




## Part C Gurobi Optimization

In [1]:
from gurobipy import *
import numpy as np

availability = np.array([
    [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1],
    [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]
])
required_shifts = [6, 2, 4, 4, 3, 3, 4, 3, 6, 3, 4, 3, 4, 4, 3]
remaining_pow = [1, 2, 0, 1, 0, 3, 1, 1, 1, 2, 1, 4, 0, 0, 1]

# Initialize model
m = Model("pediatrician_scheduling")

# Constants
NUM_PEDIATRICIANS = 15
NUM_SHIFTS = 56  # 14 shifts per week * 4 weeks
NUM_WEEKS = 4
W1, W2, W3, W4 = 0.3, 0.3, 0.3, 0.10   # Weights for soft constraints


# Decision Variables
# x[p,s]: 1 if pediatrician p works shift s, 0 otherwise
x = m.addVars(NUM_PEDIATRICIANS, NUM_SHIFTS, vtype=GRB.BINARY, name="x")

# s in python is 0-indexed, so the shifts start from 0.

# pow[p,w]: 1 if pediatrician p is POW in week w, 0 otherwise
pow = m.addVars(NUM_PEDIATRICIANS, NUM_WEEKS, vtype=GRB.BINARY, name="pow")

# Constraint a: Shift Coverage
for s in range(NUM_SHIFTS):
    m.addConstr(quicksum(x[p,s] for p in range(NUM_PEDIATRICIANS)) == 1)

# Constraint b: Required Shifts
for p in range(NUM_PEDIATRICIANS):
    m.addConstr(quicksum(x[p,s] for s in range(NUM_SHIFTS)) == required_shifts[p])

# Constraint c: Back-to-back Shift Constraints
for p in range(NUM_PEDIATRICIANS):
    for s in range(NUM_SHIFTS-1):
        m.addConstr(x[p,s] + x[p,s+1] <= 1)

# Constraint d: Consecutive Night Shift Limit
for p in range(NUM_PEDIATRICIANS):
    for s in range(0, NUM_SHIFTS-2, 2):  # Only checking night shifts
        m.addConstr(x[p,s] + x[p,s+2] <= 1)

# Constraint e: Weekend Shift Limit (When constraints c and d are satisfied, constraint e will always be met.)
# for p in range(NUM_PEDIATRICIANS):
#     for w in range(NUM_WEEKS):
#         weekend_shifts = [w*14+9, w*14+10, w*14+11, w*14+12, w*14+13]  # Friday night to Sunday night
#         m.addConstr(quicksum(x[p,s] for s in weekend_shifts) <= 2)

# Constraint f: Availability Constraints
for p in range(NUM_PEDIATRICIANS):
    for s in range(NUM_SHIFTS):
        m.addConstr(x[p,s] <= availability[s][p])

# Constraint g: POW Assignment
for w in range(NUM_WEEKS):
    m.addConstr(quicksum(pow[p,w] for p in range(NUM_PEDIATRICIANS)) == 1)

# Constraint h: POW Mandatory Shifts
for p in range(NUM_PEDIATRICIANS):
    for w in range(NUM_WEEKS):
        mandatory_shifts = [2 + w*14, 6 + w*14, 11 + w*14]  # Tuesday day, Thursday day, Saturday night (s in python is 0-indexed, so the shifts start from 0.)
        m.addConstr(quicksum(x[p,s] for s in mandatory_shifts) >= 3 * pow[p,w])

# Constraint i: POW Monthly Limits
for p in range(NUM_PEDIATRICIANS):
    m.addConstr(quicksum(pow[p,w] for w in range(NUM_WEEKS)) <= 1)

# Constraint j: POW Remaining Limits
for p in range(NUM_PEDIATRICIANS):
    m.addConstr(quicksum(pow[p,w] for w in range(NUM_WEEKS)) <= remaining_pow[p])

# Soft Constraints

# Violation variables for soft constraints
consec_weekend_violations = m.addVars(NUM_PEDIATRICIANS, NUM_WEEKS, name="consec_weekend_violations")
night_spacing_violations = m.addVars(NUM_PEDIATRICIANS, NUM_SHIFTS, name="night_spacing_violations")
evening_shift_violations = m.addVars(NUM_PEDIATRICIANS, NUM_WEEKS, name="evening_shift_violations")
day_night_balance_violations = m.addVars(NUM_PEDIATRICIANS, name="day_night_balance_violations")

# 1. No consecutive weekends
# Helper variables to indicate if a pediatrician works any shift during a weekend
weekend_work = m.addVars(NUM_PEDIATRICIANS, NUM_WEEKS, vtype=GRB.BINARY, name="weekend_work")

for p in range(NUM_PEDIATRICIANS):
    for w in range(NUM_WEEKS-1):
        # If both weekend_work variables are 1, then there must be a violation
        m.addConstr(weekend_work[p,w] + weekend_work[p,w+1] <= 1 + consec_weekend_violations[p,w])
for p in range(NUM_PEDIATRICIANS):
    for w in range(NUM_WEEKS-1):
        weekend_curr = [w*14+9, w*14+10, w*14+11, w*14+12, w*14+13]
        weekend_next = [(w+1)*14+9, (w+1)*14+10, (w+1)*14+11, (w+1)*14+12, (w+1)*14+13]
        m.addConstr(
            quicksum(x[p,s] for s in weekend_curr) + 
            quicksum(x[p,s] for s in weekend_next) <= 
            2 + consec_weekend_violations[p,w]
        )
        
# For each pediatrician and week, set weekend_work to 1 if they work any shift that weekend
for p in range(NUM_PEDIATRICIANS):
    for w in range(NUM_WEEKS):
        weekend_shifts = [w*14+9, w*14+10, w*14+11, w*14+12, w*14+13]  # Friday night to Sunday night
        
        # weekend_work[p,w] should be 0 if no weekend shift is worked 
        # The minization process will make weekend_work[p,w]=0 when it is possible.
        # m.addConstr(weekend_work[p,w] <= quicksum(x[p,s] for s in weekend_shifts)) 
        
        for s in weekend_shifts:
            m.addConstr(weekend_work[p,w] >= x[p,s])


# 2. Night shift spacing
for p in range(NUM_PEDIATRICIANS):
    for s in range(NUM_SHIFTS-4):
        if s % 2 == 0:  # Only for night shifts. % means modulo which is the remainder of the division. 
            m.addConstr(x[p,s] + x[p,s+4] <= 1 + night_spacing_violations[p,s])

# 3. Evening shift limit
for p in range(NUM_PEDIATRICIANS):
    for w in range(NUM_WEEKS):
        evening_shifts = [s for s in range(w*14, (w+1)*14) if s % 2 == 1]
        m.addConstr(
            quicksum(x[p,s] for s in evening_shifts) <= 
            2 + evening_shift_violations[p,w]
        )

# 4. Day/Night balance
for p in range(NUM_PEDIATRICIANS):
    day_shifts = [s for s in range(NUM_SHIFTS) if s % 2 == 0]
    night_shifts = [s for s in range(NUM_SHIFTS) if s % 2 == 1]
    m.addConstr(
        quicksum(x[p,s] for s in day_shifts) - 
        quicksum(x[p,s] for s in night_shifts) <= 
        day_night_balance_violations[p]
    )
    m.addConstr(
        quicksum(x[p,s] for s in night_shifts) - 
        quicksum(x[p,s] for s in day_shifts) <= 
        day_night_balance_violations[p]
    )

# Objective Function
obj = (
    W1* quicksum(consec_weekend_violations[p,w] for p in range(NUM_PEDIATRICIANS) for w in range(NUM_WEEKS)) +
    W2 * quicksum(night_spacing_violations[p,s] for p in range(NUM_PEDIATRICIANS) for s in range(NUM_SHIFTS)) +
    W3 * quicksum(evening_shift_violations[p,w] for p in range(NUM_PEDIATRICIANS) for w in range(NUM_WEEKS)) +
    W4 * quicksum(day_night_balance_violations[p] for p in range(NUM_PEDIATRICIANS))
)

m.setObjective(obj, GRB.MINIMIZE)

# Optimize the model
m.optimize()

# Function to print the solution
def print_solution():
    if m.status == GRB.OPTIMAL or m.status == GRB.TIME_LIMIT:
        print("\nSchedule:")
        for w in range(NUM_WEEKS):
            print(f"\nWeek {w+1}:")
            print("POW:", end=" ")
            for p in range(NUM_PEDIATRICIANS):
                if pow[p,w].x > 0.5:
                    print(f"Pediatrician {p+1}")
            
            for d in range(7):
                day_shift = w*14 + d*2
                night_shift = day_shift + 1
                print(f"Day {d+1}:")
                print(f"  Day shift: ", end="")
                for p in range(NUM_PEDIATRICIANS):
                    if x[p,day_shift].x > 0.5:
                        print(f"Pediatrician {p+1}")
                print(f"  Night shift: ", end="")
                for p in range(NUM_PEDIATRICIANS):
                    if x[p,night_shift].x > 0.5:
                        print(f"Pediatrician {p+1}")
        
        print("\nObjective Value:", m.objVal)
        # Print a list of what the number of each violation
    else:
        print("No solution found")
    print(f"Consecutive Weekend Violation: {quicksum(consec_weekend_violations[p,w].x for p in range(NUM_PEDIATRICIANS) for w in range(NUM_WEEKS))} times")
    print(f"Night Spacing Violation: {quicksum(night_spacing_violations[p,s].x for p in range(NUM_PEDIATRICIANS) for s in range(NUM_SHIFTS))} times")
    print(f"Evening Shift Violation: {quicksum(evening_shift_violations[p,w].x for p in range(NUM_PEDIATRICIANS) for w in range(NUM_WEEKS))} times")
    print(f"Day_Night Balance Violation: {quicksum(day_night_balance_violations[p].x for p in range(NUM_PEDIATRICIANS))} times")
    for p in range(NUM_PEDIATRICIANS):
        if day_night_balance_violations[p].x > 0.5:
            print(f"Pediatrician {p+1} has a day/night balance violation")
    print()
# Print the solution
print_solution()



Set parameter Username
Academic license - for non-commercial use only - expires 2025-11-20
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 24.2.0 24C5079e)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3105 rows, 1935 columns and 9990 nonzeros
Model fingerprint: 0xa54b7908
Variable types: 975 continuous, 960 integer (960 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e-01, 3e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Presolve removed 2285 rows and 1024 columns
Presolve time: 0.03s
Presolved: 820 rows, 911 columns, 5623 nonzeros
Variable types: 0 continuous, 911 integer (804 binary)
Found heuristic solution: objective 4.7000000

Root relaxation: objective 1.200000e+00, 336 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntIn

### Pediatrician Schedule

#### Week 1
**POW:** Pediatrician 1

| Day     | Day Shift           | Night Shift         |
|---------|---------------------|---------------------|
| Day 1   | Pediatrician 13     | Pediatrician 4      |
| Day 2   | Pediatrician 1      | Pediatrician 8      |
| Day 3   | Pediatrician 3      | Pediatrician 13     |
| Day 4   | Pediatrician 1      | Pediatrician 12     |
| Day 5   | Pediatrician 5      | Pediatrician 1      |
| Day 6   | Pediatrician 11     | Pediatrician 1      |
| Day 7   | Pediatrician 12     | Pediatrician 10     |

#### Week 2
**POW:** Pediatrician 9

| Day     | Day Shift           | Night Shift         |
|---------|---------------------|---------------------|
| Day 1   | Pediatrician 15     | Pediatrician 3      |
| Day 2   | Pediatrician 9      | Pediatrician 7      |
| Day 3   | Pediatrician 8      | Pediatrician 11     |
| Day 4   | Pediatrician 9      | Pediatrician 5      |
| Day 5   | Pediatrician 6      | Pediatrician 2      |
| Day 6   | Pediatrician 15     | Pediatrician 9      |
| Day 7   | Pediatrician 2      | Pediatrician 8      |

#### Week 3
**POW:** Pediatrician 7

| Day     | Day Shift           | Night Shift         |
|---------|---------------------|---------------------|
| Day 1   | Pediatrician 1      | Pediatrician 3      |
| Day 2   | Pediatrician 7      | Pediatrician 1      |
| Day 3   | Pediatrician 14     | Pediatrician 9      |
| Day 4   | Pediatrician 7      | Pediatrician 14     |
| Day 5   | Pediatrician 13     | Pediatrician 10     |
| Day 6   | Pediatrician 6      | Pediatrician 7      |
| Day 7   | Pediatrician 10     | Pediatrician 6      |

#### Week 4
**POW:** Pediatrician 4

| Day     | Day Shift           | Night Shift         |
|---------|---------------------|---------------------|
| Day 1   | Pediatrician 12     | Pediatrician 9      |
| Day 2   | Pediatrician 4      | Pediatrician 5      |
| Day 3   | Pediatrician 9      | Pediatrician 15     |
| Day 4   | Pediatrician 4      | Pediatrician 13     |
| Day 5   | Pediatrician 3      | Pediatrician 11     |
| Day 6   | Pediatrician 14     | Pediatrician 4      |
| Day 7   | Pediatrician 11     | Pediatrician 14     |

---

#### Violations
- **Objective Value:** 1.80 
- **Consecutive Weekend Violation:** 0.0 times  
- **Night Spacing Violation:** 4.0 times  
- **Evening Shift Violation:** 0.0 times  
- **Day/Night Balance Violation:** 6.0 times  

##### Pediatricians with Day/Night Balance Violations
- Pediatrician 5  
- Pediatrician 6  
- Pediatrician 8  
- Pediatrician 10  
- Pediatrician 12  
- Pediatrician 15

## Part D. Discussion and Recommendation

If Gurobi does not return a feasible solution, it suggests that the constraints are overly restrictive. We recommend Meg take the following steps:

**1. Change Hard Constraints to Soft Constraints:**

- **Required Number of Shifts (Constraint b):** Change the equality (=) to a greater-than-or-equal-to (≥) condition, allowing pediatricians to work more than the required shifts. Add an overtime penalty to the objective function to make it a soft constraint.
- **Consecutive Night Shifts (Constraint d):** Permit consecutive night shifts for pediatricians but keep the Back-to-Back Shifts (Constraint c) as a hard constraint to ensure no risk of over-working.
- **Weekend Shift Limit (Constraint e):** Allow up to three weekend shifts occasionally, with penalties for exceeding two.
- **POW Mandatory Shifts (Constraint h):** Relax mandatory shifts for POW pediatricians and add penalties for missing these shifts.
- **POW Monthly Limits (Constraint i):** Allow pediatricians to serve as POW more than once in a cycle, with penalties for exceeding limits.

**2. Adjust Penalty Weights:** Meg could prioritize constraints by assigning higher penalty weights to soft constraints that were converted from hard constraints.

**3. Maintain Key Hard Constraints:**

In additional to above hard constraints which we believe are flexible to adjust, the following hard constraints are recommended to keep as hard constraints to satisfy workplace safety and organization requirement.

- **Shift Coverage (Constraint a):** Every shift must be covered by exactly one pediatrician.
- **Back-to-Back Shifts (Constraint c):** Pediatricians should not work for back-to-back shifts.
- **Availability Requests (Constraint f):** Pediatricians cannot be scheduled during requested time off.
- **POW Assignment Requirement (Constraint g):** Ensure one pediatrician is assigned as POW each week.
- **POW Remaining Limits (Constraint j):** POW limits should remain as a hard constraint to ensure equal treatment for all pediatricians.

If relaxing constraints still does not yield a feasible solution, Meg should consider **hiring additional workforce**.

